In [4]:
import requests
from bs4 import BeautifulSoup, NavigableString
import pandas as pd
import time

def scrape_wos_corrected(url):
    print(f"正在抓取: {url}")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response.encoding = 'utf-8'
    except Exception as e:
        print(f"请求错误: {e}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 结果列表
    journal_data = []
    
    # 获取 body 内容
    body_content = soup.find('body')
    if not body_content:
        return None

    # 遍历所有文本节点
    # 这里的逻辑是：按顺序读取网页上的每一段文字
    for element in body_content.descendants:
        
        # 只处理文本内容，忽略换行和空字符
        if isinstance(element, NavigableString):
            text = element.strip()
            if not text:
                continue
            
            # 获取当前文本的父标签名称 (比如 'b', 'dd', 'dt', 'p')
            parent_tag = element.parent.name
            
            # === 核心修正逻辑 ===
            # 判定为【缩写】的情况：
            # 1. 父标签是 <b> 或 <strong> (粗体)
            # 2. 父标签是 <dd> (定义描述，通常自带缩进，这是最可能漏掉的情况)
            is_abbreviation = parent_tag in ['b', 'strong', 'dd']
            
            # 判定为【全称】的情况：
            # 父标签是 <dt> (定义术语) 或 <p>, <div>, <body> 等普通容器
            # 且排除掉显然不是标题的短词（可选）
            is_full_title = parent_tag in ['dt', 'p', 'div', 'body', 'li']

            if is_abbreviation:
                # 如果判定为缩写，且列表中有上一条记录，则更新上一条记录
                if journal_data:
                    # 如果之前已经是 NA，则覆盖；如果之前已经有缩写了（罕见情况），可以用分号拼接
                    if journal_data[-1]['Abbreviation'] == 'NA':
                         journal_data[-1]['Abbreviation'] = text
                    else:
                         # 防止同一个缩写被拆成两段文本
                         journal_data[-1]['Abbreviation'] += " " + text
            
            elif is_full_title:
                # 如果判定为全称，则新增一条记录
                # 过滤掉页面导航的干扰词（比如单独的字母 A, B, C...）
                if len(text) < 2 and text.isalpha():
                    continue 

                journal_data.append({
                    'Full Title': text,
                    'Abbreviation': 'NA' # 默认先填 NA
                })

    df = pd.DataFrame(journal_data)
    return df

# === 运行抓取 ===
# 你可以把下面的 URL 里的 A 换成其他字母来测试
target_url = "https://wos-help.webofscience.com/WOKRS535R111/help/WOS/A_abrvjt.html"

df = scrape_wos_corrected(target_url)

if df is not None:
    # 打印前10行看看效果
    print("预览前 10 行数据：")
    print(df.head(10))
    
    # 导出
    df.to_excel("WoS_Journal_Abbreviations_Fixed.xlsx", index=False)
    print("\n文件已保存: WoS_Journal_Abbreviations_Fixed.xlsx")

正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/A_abrvjt.html
预览前 10 行数据：
                                          Full Title         Abbreviation
0  This list shows the abbreviations used for jou...                   NA
1  (boldface) title from this list and paste     ...                   NA
2  Use the cited work index to find additional ab...                   NA
3                              of the cited works in                   NA
4                                                  .                   NA
5  Click on a letter to move through the journal ...         Journal List
6                    A + U-ARCHITECTURE AND URBANISM     A U-ARCHIT URBAN
7  A CRITICAL REVIEW: LASER TECHNOLOGIES FOR DEFE...  P SOC PHOTO-OPT INS
8          A KALEIDOSCOPIC VIEW OF NATURAL RESOURCES                   NA
9                          A MIDSUMMER NIGHT'S DREAM     SHAKESPEARE SURV

文件已保存: WoS_Journal_Abbreviations_Fixed.xlsx


In [5]:
# === A-Z 全量爬取脚本 ===
import pandas as pd
import time

all_dfs = []
letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ" # 遍历 A 到 Z

print("开始全量爬取...")

for char in letters:
    url = f"https://wos-help.webofscience.com/WOKRS535R111/help/WOS/{char}_abrvjt.html"
    print(f"正在处理字母: {char} ...")
    
    # 复用上面的函数
    df_part = scrape_wos_corrected(url)
    
    if df_part is not None and not df_part.empty:
        all_dfs.append(df_part)
    
    # 暂停 1 秒，防止请求过快被封
    time.sleep(1)

# 合并并保存
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_excel("WoS_All_Journals.xlsx", index=False)
    print(f"全部完成！共抓取 {len(final_df)} 条数据。已保存为 WoS_All_Journals.xlsx")

开始全量爬取...
正在处理字母: A ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/A_abrvjt.html
正在处理字母: B ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/B_abrvjt.html
正在处理字母: C ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/C_abrvjt.html
正在处理字母: D ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/D_abrvjt.html
正在处理字母: E ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/E_abrvjt.html
正在处理字母: F ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/F_abrvjt.html
正在处理字母: G ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/G_abrvjt.html
正在处理字母: H ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/H_abrvjt.html
正在处理字母: I ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/I_abrvjt.html
正在处理字母: J ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/J_abrvjt.html
正在处理字母: K ...
正在抓取: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/K_abrvjt.html
